# Notebook1 - Time Series Features

## 學習目標

本 notebook 將 raw RRD-like metrics 轉換成可用於異常偵測的時間序列特徵。

對應教學設計：
- timestamp index 與欄位型態整理
- resampling / missing value 檢查
- rolling statistics
- lag features
- rate / ratio features
- inbound vs outbound comparison

## Step 0 - 環境與輸入資料

輸入：`data/synthetic/synthetic_rrd_metrics.csv`

輸出：`data/processed/features.csv`

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

def show_fig(fig):
    display(fig)
    plt.close(fig)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path("..").resolve()

DATA_SYNTHETIC = PROJECT_ROOT / "data" / "synthetic"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS = PROJECT_ROOT / "reports"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
raw_path = DATA_SYNTHETIC / "synthetic_rrd_metrics.csv"
event_path = DATA_SYNTHETIC / "synthetic_event_catalog.csv"

df = pd.read_csv(raw_path, parse_dates=["timestamp"])
event_catalog = pd.read_csv(event_path, parse_dates=["start_time", "end_time"])
df = df.sort_values(["device_id", "port_id", "timestamp"]).reset_index(drop=True)

print(f"rows: {len(df):,}")
print(f"ports: {df['port_id'].nunique()}")
print(f"time range: {df['timestamp'].min()} -> {df['timestamp'].max()}")
display(df.head())
display(event_catalog)

## Step 1 - 資料品質檢查

確認每個 port 的時間解析度、缺值比例與事件標籤分布。

In [ ]:
metric_cols = [
    "INOCTETS", "OUTOCTETS", "INERRORS", "OUTERRORS",
    "INUCASTPKTS", "OUTUCASTPKTS", "INNUCASTPKTS", "OUTNUCASTPKTS",
    "INDISCARDS", "OUTDISCARDS", "INUNKNOWNPROTOS",
    "INBROADCASTPKTS", "OUTBROADCASTPKTS", "INMULTICASTPKTS", "OUTMULTICASTPKTS",
]

quality = (
    df.groupby(["device_id", "port_id"])
    .agg(
        rows=("timestamp", "size"),
        start=("timestamp", "min"),
        end=("timestamp", "max"),
        event_rows=("event_id", lambda s: (s.astype(str) != "").sum()),
    )
    .reset_index()
)
missing = df[metric_cols].isna().mean().rename("missing_rate").reset_index().rename(columns={"index": "metric"})
display(quality)
display(missing)

## Step 2 - 建立監控語意特徵

將原始欄位轉成總量、比例與 rate。這些特徵會在後續 notebook 中重複使用。

In [ ]:
def safe_divide(a, b):
    return np.divide(a, b, out=np.zeros_like(a, dtype=float), where=np.asarray(b) != 0)


df["traffic_total"] = df["INOCTETS"] + df["OUTOCTETS"]
df["ucast_total"] = df["INUCASTPKTS"] + df["OUTUCASTPKTS"]
df["nucast_total"] = df["INNUCASTPKTS"] + df["OUTNUCASTPKTS"]
df["errors_total"] = df["INERRORS"] + df["OUTERRORS"]
df["discards_total"] = df["INDISCARDS"] + df["OUTDISCARDS"]
df["broadcast_total"] = df["INBROADCASTPKTS"] + df["OUTBROADCASTPKTS"]
df["multicast_total"] = df["INMULTICASTPKTS"] + df["OUTMULTICASTPKTS"]
df["unknown_total"] = df["INUNKNOWNPROTOS"]

df["avg_packet_size"] = safe_divide(df["traffic_total"].to_numpy(), df["ucast_total"].to_numpy())
df["error_rate"] = safe_divide(df["errors_total"].to_numpy(), df["ucast_total"].to_numpy())
df["discard_rate"] = safe_divide(df["discards_total"].to_numpy(), df["ucast_total"].to_numpy())
df["broadcast_ratio"] = safe_divide(df["broadcast_total"].to_numpy(), df["ucast_total"].to_numpy())
df["multicast_ratio"] = safe_divide(df["multicast_total"].to_numpy(), df["ucast_total"].to_numpy())
df["in_out_octets_ratio"] = safe_divide(df["INOCTETS"].to_numpy(), df["OUTOCTETS"].to_numpy())

feature_preview = [
    "timestamp", "device_id", "port_id", "event_label",
    "traffic_total", "ucast_total", "avg_packet_size",
    "error_rate", "discard_rate", "broadcast_ratio", "multicast_ratio",
]
display(df[feature_preview].head())
display(df[["traffic_total", "ucast_total", "avg_packet_size", "error_rate", "discard_rate"]].describe())

## Step 3 - 視覺化：流量、封包與錯誤/丟棄

這對應 Grafana 的流量 trend、封包 trend、error / discard rate panels。

In [ ]:
sample_port = "port-id7427"
plot_df = df[df["port_id"] == sample_port].copy()

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
axes[0].plot(plot_df["timestamp"], plot_df["INOCTETS"], label="INOCTETS", linewidth=1)
axes[0].plot(plot_df["timestamp"], plot_df["OUTOCTETS"], label="OUTOCTETS", linewidth=1)
axes[0].set_title(f"{sample_port} - In / Out Octets")
axes[0].legend(loc="upper left")

axes[1].plot(plot_df["timestamp"], plot_df["ucast_total"], color="tab:green", label="ucast_total", linewidth=1)
axes[1].set_title("Unicast Packets")
axes[1].legend(loc="upper left")

axes[2].plot(plot_df["timestamp"], plot_df["error_rate"], label="error_rate", linewidth=1)
axes[2].plot(plot_df["timestamp"], plot_df["discard_rate"], label="discard_rate", linewidth=1)
axes[2].set_title("Error / Discard Rate")
axes[2].legend(loc="upper left")

for ax in axes:
    for _, event in event_catalog.iterrows():
        if event["port_id"] in [sample_port, "MULTI"]:
            ax.axvspan(event["start_time"], event["end_time"], alpha=0.10, color="tab:red")
    ax.grid(alpha=0.25)

plt.tight_layout()
show_fig(fig)

## Step 4 - Rolling statistics 與 lag features

建立 1 小時 rolling 與 1 天 p95 / MAD，支援 baseline、SPC 與 ML。

In [ ]:
def rolling_mad(series):
    median = np.median(series)
    return np.median(np.abs(series - median))


base_features = [
    "traffic_total", "ucast_total", "errors_total", "discards_total",
    "broadcast_total", "multicast_total", "unknown_total",
    "avg_packet_size", "error_rate", "discard_rate", "broadcast_ratio", "multicast_ratio",
]

feature_frames = []
for (_, port_id), g0 in df.groupby(["device_id", "port_id"], sort=False):
    g = g0.sort_values("timestamp").set_index("timestamp").copy()
    generated = {}
    for col in base_features:
        generated[f"{col}_rolling_mean_1h"] = g[col].rolling("60min", min_periods=3).mean()
        generated[f"{col}_rolling_std_1h"] = g[col].rolling("60min", min_periods=3).std()
        generated[f"{col}_rolling_median_1h"] = g[col].rolling("60min", min_periods=3).median()
        generated[f"{col}_rolling_p95_1d"] = g[col].rolling("1d", min_periods=12).quantile(0.95)
        generated[f"{col}_rolling_mad_1d"] = g[col].rolling("1d", min_periods=12).apply(rolling_mad, raw=True)
        generated[f"{col}_lag_5m"] = g[col].shift(1)
        generated[f"{col}_lag_1h"] = g[col].shift(12)
    g = pd.concat([g, pd.DataFrame(generated, index=g.index)], axis=1)
    feature_frames.append(g.reset_index())

features = pd.concat(feature_frames, ignore_index=True).bfill().fillna(0)
print(f"feature columns: {len(features.columns)}")
display(features[["timestamp", "port_id", "traffic_total", "traffic_total_rolling_mean_1h", "traffic_total_rolling_p95_1d"]].head())

## Step 5 - 視覺化：rolling baseline 與 avg packet size

觀察大量小封包與大檔案傳輸時，`avg_packet_size` 如何改變。

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
p = features[features["port_id"] == "port-id7428"].copy()
axes[0].plot(p["timestamp"], p["traffic_total"], label="traffic_total", linewidth=1)
axes[0].plot(p["timestamp"], p["traffic_total_rolling_mean_1h"], label="rolling_mean_1h", linewidth=1.4)
axes[0].plot(p["timestamp"], p["traffic_total_rolling_p95_1d"], label="rolling_p95_1d", linewidth=1.2)
axes[0].set_title("Traffic vs rolling baselines")
axes[0].legend(loc="upper left")

axes[1].plot(p["timestamp"], p["avg_packet_size"], label="avg_packet_size", color="tab:purple", linewidth=1)
axes[1].plot(p["timestamp"], p["avg_packet_size_rolling_median_1h"], label="rolling_median_1h", color="tab:orange", linewidth=1.2)
axes[1].set_title("Average packet size")
axes[1].legend(loc="upper left")

for ax in axes:
    for _, event in event_catalog.iterrows():
        if event["port_id"] in ["port-id7428", "MULTI"]:
            ax.axvspan(event["start_time"], event["end_time"], alpha=0.10, color="tab:red")
    ax.grid(alpha=0.25)
plt.tight_layout()
show_fig(fig)

## Step 6 - 輸出 features.csv

In [ ]:
out_path = DATA_PROCESSED / "features.csv"
features.to_csv(out_path, index=False)
print(f"saved: {out_path}")

## 小結

本 notebook 完成 Metrics → Features 的轉換。下一步會用這些 rolling baseline、ratio 與 rate 建立 rule-based anomaly flags。